In [5]:
import pandas as pd

In [1]:
import sqlite3
# sqlite3 ships with Python's standard library — no pip install needed

In [2]:
from sqlalchemy import create_engine

engine = create_engine('sqlite:///data/db_delays.db')

In [7]:
# --- 2. Load cleaned CSVs ---
dim_station = pd.read_csv('data/processed/dim_station.csv')
dim_line = pd.read_csv('data/processed/dim_line.csv')
dim_date = pd.read_csv('data/processed/dim_date.csv', parse_dates=['date'])
fact_delays = pd.read_csv('data/processed/fact_delays.csv', parse_dates=[
    'arrival_plan', 'departure_plan', 'arrival_change', 'departure_change'
])
print("All tables loaded into SQLite")

All tables loaded into SQLite


In [11]:
dim_station.to_sql('dim_station', engine, if_exists='replace', index=False)
dim_line.to_sql('dim_line', engine, if_exists='replace', index=False)
dim_date.to_sql('dim_date', engine, if_exists='replace', index=False)

9

In [12]:
# make sure column names are correct
dim_station = dim_station.rename(columns={'lat': 'latitude', 'long': 'longitude'})


In [13]:
# reapply the deduplication step from before
fact_delays_sorted = fact_delays.sort_values(
    by=['stop_id', 'arrival_change', 'departure_change'],
    na_position='first'
)
fact_delays_dedup = fact_delays_sorted.drop_duplicates(subset='stop_id', keep='last')

print(f"fact_delays: {len(fact_delays)} → fact_delays_dedup: {len(fact_delays_dedup)}")

fact_delays: 2061357 → fact_delays_dedup: 2029894


In [14]:
engine = create_engine('sqlite:///data/db_delays.db')

dim_station.to_sql('dim_station', engine, if_exists='replace', index=False)
dim_line.to_sql('dim_line', engine, if_exists='replace', index=False)
dim_date.to_sql('dim_date', engine, if_exists='replace', index=False)
fact_delays_dedup.to_sql('fact_delays', engine, if_exists='replace', index=False, chunksize=5000)

print("All tables loaded into SQLite")

All tables loaded into SQLite


In [15]:
# save fact table
fact_delays_dedup.to_csv('data/processed/fact_delays_dedup.csv', index=False)

In [16]:
print(pd.read_sql('SELECT COUNT(*) FROM dim_station', engine))

   COUNT(*)
0      1996


In [17]:
print(pd.read_sql('SELECT COUNT(*) FROM dim_line', engine))

   COUNT(*)
0       296


In [18]:
print(pd.read_sql('SELECT COUNT(*) FROM dim_date', engine))

   COUNT(*)
0         9


In [19]:
query = """
SELECT s.station, s.state, AVG(f.arrival_delay_m) AS avg_delay, COUNT(*) AS n_stops
FROM fact_delays f
JOIN dim_station s ON f.station_id = s.station_id
GROUP BY s.station, s.state
HAVING COUNT(*) >= 20
ORDER BY avg_delay DESC
LIMIT 15
"""
pd.read_sql(query, engine)

,station,state,avg_delay,n_stops
0,Steinau (Straße),Hessen,6.169811,265
1,Flieden,Hessen,5.785408,233
2,Langenselbold,Hessen,5.287846,469
3,St. Goar,Rheinland-Pfalz,5.120370,324
4,Neuhof (Kr Fulda),Hessen,4.993243,296
5,Bad Soden-Salmünster,Hessen,4.975232,323
6,Schlüchtern,Hessen,4.791005,378
7,Rodenbach bei Hanau,Hessen,4.688525,244
8,Sonthofen,Bayern,4.687773,458
9,Teisendorf,Bayern,4.666667,297
